<a href="https://colab.research.google.com/github/AlanAlbertoJaimesV/everpeak-analysis/blob/main/EverPeak.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =====================================================================
# CELDA 1: Importación de librerías y carga de datos
# =====================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar los datos limpios
df = pd.read_csv("/datasets/everpeak_clean.csv")

# =====================================================================
# CELDA 2: Visión General y Estadísticas Descriptivas
# =====================================================================
print("--- Información General ---")
df.info()

print("\n--- Estadísticas Descriptivas ---")
columnas_numericas = ["price", "quantity", "order_value", "customer_age"]
display(df[columnas_numericas].describe())

# =====================================================================
# CELDA 3: Visualización Diagnóstica (Detectando la forma del negocio)
# =====================================================================
# Graficar histogramas con la curva KDE para ver la distribución real
for col in columnas_numericas:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col], bins=50, color='skyblue', kde=True)
    plt.title(f'Distribución de la variable: {col}')
    plt.show()

# =====================================================================
# CELDA 4: Tratamiento de Outliers (Winsorización)
# =====================================================================
# Capamos los valores atípicos reales al percentil 99 para no distorsionar
# el análisis del cliente promedio sin borrar los ingresos de los VIP.

for col in ["price", "quantity", "order_value"]:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    # Creamos las nuevas columnas capadas
    df[f'{col}_winsor'] = np.clip(df[col], lower, upper)

print("\n--- Estadísticas Post-Winsorización (Cap) ---")
display(df[["order_value", "order_value_winsor"]].describe())

# =====================================================================
# CELDA 5: Feature Engineering (Segmentación de Clientes por Gasto y Edad)
# =====================================================================
def classify_segment(row):
    age = row['customer_age']
    spend = row['order_value']

    # Manejo de valores faltantes
    if pd.isna(age) or pd.isna(spend):
        return "Error en Datos"

    # Segmentación de Alto Valor (Gasto >= 10000)
    if spend >= 10000:
        if age >= 55: return "Senior VIP"
        else: return "Junior VIP"

    # Segmentación de Valor Medio (Gasto entre 5000 y 9999)
    elif spend >= 5000:
        if age >= 55: return "Sr. Medium Value"
        else: return "Jr. Medium Value"

    # Segmentación de Valor Bajo (Gasto < 5000)
    else:
        return "Low Value"

# Aplicamos la función fila por fila
df["customer_segment"] = df.apply(classify_segment, axis=1)

# =====================================================================
# CELDA 6: Feature Engineering (Segmentación por Volumen de Compra)
# =====================================================================
def classify_volume(row):
    age = row['customer_age']
    qty = row['quantity']

    if pd.isna(age) or pd.isna(qty):
        return "Error en Datos"

    if qty > 22:
        if age > 55: return "Sr. High Volume"
        else: return "Jr. High Volume"
    elif qty <= 22:
        if age > 55: return "Sr. Low Volume"
        else: return "Jr. Low Volume"

# Aplicamos la función
df["volume_segment"] = df.apply(classify_volume, axis=1)

# =====================================================================
# CELDA 7: Resultados Ejecutivos Finales
# =====================================================================
print("\n--- Resultados de Segmentación de Clientes ---")
print(df['customer_segment'].value_counts())

print("\n--- Resultados de Segmentación por Volumen ---")
print(df['volume_segment'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '/datasets/everpeak_clean.csv'